In [ ]:
import os
import time
import random
import logging
import hashlib
import csv
from typing import Optional, List, Dict, Any
from urllib.parse import urljoin

# Use curl_cffi for better WAF bypass (mimics real browser TLS fingerprints)
from curl_cffi import requests
from bs4 import BeautifulSoup
import pandas as pd

# --- Setup Logging ---
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("scraper.log"),
        logging.StreamHandler()
    ],
    force=True
)
logger = logging.getLogger(__name__)

In [ ]:
# --- Configuration & Guardrails ---
# Scrape years 2015 to 2026
START_YEAR = 2015
END_YEAR = 2026 # Set to 2015 for testing. Change back to 2026 for full run.

# Output Configuration
# New specific output folder
OUTPUT_FOLDER = r"C:\Users\shave\OneDrive\Desktop\Football Recruit Data"

if not os.path.exists(OUTPUT_FOLDER):
    try:
        os.makedirs(OUTPUT_FOLDER)
        print(f"Created folder: {OUTPUT_FOLDER}")
    except OSError as e:
        # Fallback if path is invalid or permission denied
        OUTPUT_FOLDER = os.getcwd()
        print(f"Error creating custom path: {e}")
        print(f"Using current working directory: {OUTPUT_FOLDER}")
else:
    print(f"Target folder exists: {OUTPUT_FOLDER}")


CACHE_DIR = "247_cache"

# 247Sports is stricter with User-Agents. We need a realistic one.
USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"

# Rate Limiting (Politeness settings)
MIN_DELAY = 1.0  # Seconds
MAX_DELAY = 3.0  # Seconds

# Crawler settings
MAX_PLAYERS_PER_YEAR = 10000 # Stop after gathering this many players per year
MAX_PAGES = 100  # Set high enough to accomodate 10k players (approx 200 pages @ 50/page)
SAVE_BATCH_SIZE = 1000

print(f"Target Years: {START_YEAR} - {END_YEAR}")
print(f"Limit: {MAX_PLAYERS_PER_YEAR} players per year.")

Target folder exists: C:\Users\shave\OneDrive\Desktop\Football Recruit Data
Target Years: 2015 - 2026
Limit: 10000 players per year.


In [ ]:
class RobustScraper:
    def __init__(self, base_url: str, cache_dir: str, year: int):
        self.base_url = base_url
        self.cache_dir = cache_dir
        self.year = year
        self.session = self._init_session()

        # Ensure cache directory exists
        os.makedirs(self.cache_dir, exist_ok=True)

    def _init_session(self):
        """
        Initialize curl_cffi session.
        Authentication/Retries handled manually in fetch loop.
        """
        session = requests.Session()
        return session

    def _get_cache_path(self, url: str) -> str:
        """Generate a consistent filename for a URL using MD5 hash."""
        url_hash = hashlib.md5(url.encode("utf-8")).hexdigest()
        return os.path.join(self.cache_dir, f"{url_hash}.html")

    def _sleep_randomly(self):
        """Sleep for a random interval to mimic human behavior."""
        sleep_time = random.uniform(MIN_DELAY, MAX_DELAY)
        time.sleep(sleep_time)

    def fetch_page(self, url: str, use_cache: bool = True) -> Optional[str]:
        cache_path = self._get_cache_path(url)

        # 1. Try Cache
        if use_cache and os.path.exists(cache_path):
            try:
                if os.path.getsize(cache_path) > 0:
                    with open(cache_path, "r", encoding="utf-8") as f:
                        return f.read()
            except Exception as e:
                logger.warning(f"Failed to read cache for {url}: {e}")

        # 2. Network Request with Manual Retry
        max_retries = 3
        for attempt in range(max_retries):
            try:
                self._sleep_randomly()

                # impersonate="chrome" sends the TLS fingerprint of a real Chrome browser
                # Adding headers to force Desktop view, which might help avoid the "Load More" mobile button issue
                headers = {
                    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
                    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
                    "Upgrade-Insecure-Requests": "1"
                }

                response = self.session.get(url, impersonate="chrome", timeout=30, headers=headers)

                if response.status_code == 403:
                    logger.warning(f"Attempt {attempt+1}/{max_retries}: 403 Forbidden. TLS fingerprint might need rotation.")
                    time.sleep(5) # Wait a bit longer
                    continue

                if response.status_code != 200:
                    logger.warning(f"Attempt {attempt+1}/{max_retries}: Status {response.status_code}")
                    continue

                content = response.text

                # 3. Save to Cache
                try:
                    with open(cache_path, "w", encoding="utf-8") as f:
                        f.write(content)
                except Exception as e:
                    logger.error(f"Failed to write cache for {url}: {e}")

                return content

            except Exception as e:
                logger.error(f"Request failed for {url}: {e}")
                time.sleep(2)

        return None

    def _safe_extract(self, element, selectors: List[str], attr: str = None, clean: bool = True) -> Any:
        if not element:
            return None
        for selector in selectors:
            try:
                found = element.select_one(selector)
                if found:
                    if attr:
                        val = found.get(attr)
                    else:
                        val = found.get_text()
                    if clean and isinstance(val, str):
                        return val.strip()
                    return val
            except Exception:
                continue
        return None

    def parse_recruit_element(self, element) -> Dict[str, Any]:
        """Parses a single recruit list item from 247Sports."""
        rank_text = self._safe_extract(element, [".primary", ".rank-column .primary"])

        # FIX: Clean rank text (remove trend indicators like "38 1" -> "38")
        if rank_text:
            rank_text = rank_text.split()[0]

        name = self._safe_extract(element, [".rankings-page__name-link", "a.name"])
        rel_url = self._safe_extract(element, [".rankings-page__name-link", "a.name"], attr="href")
        abs_url = urljoin(self.base_url, rel_url) if rel_url else None

        # Metrics
        metrics = self._safe_extract(element, [".metrics", ".score-column .metrics"])
        height, weight = None, None
        if metrics and "/" in metrics:
            parts = metrics.split("/")
            if len(parts) >= 2:
                raw_ht = parts[0].strip()
                raw_wt = parts[1].strip()

                # HT: 6-2.5 -> 6' 2.5"
                if "-" in raw_ht:
                    h_parts = raw_ht.split("-")
                    if len(h_parts) == 2:
                        height = f"{h_parts[0]}' {h_parts[1]}\""
                    else:
                        height = raw_ht
                else:
                    height = raw_ht

                # WT: 210 -> 210 lbs
                weight = f"{raw_wt} lbs"

        position = self._safe_extract(element, [".position"])

        # --- Location Parsing ---
        raw_location = self._safe_extract(element, [".meta"])
        high_school, city, state = None, None, None

        if raw_location:
            # Expected format: "High School (City, ST)"
            if "(" in raw_location and ")" in raw_location:
                parts = raw_location.split("(")
                high_school = parts[0].strip()

                loc_part = parts[1].replace(")", "").strip()
                if "," in loc_part:
                    loc_split = loc_part.split(",")
                    if len(loc_split) >= 2:
                        city = loc_split[0].strip()
                        state = loc_split[1].strip()
                    else:
                        city = loc_part
                else:
                    city = loc_part
            else:
                # Fallback if format doesn't match expected "School (City, ST)"
                high_school = raw_location

        rating = self._safe_extract(element, [".score", ".rating"])

        school_element = element.select_one(".status img")
        committed_school = school_element.get("alt") if school_element else "Uncommitted"
        if not committed_school or committed_school == "":
             committed_school = self._safe_extract(element, [".status a"])

        return {
            "year": self.year, # Add year to data
            "rank": rank_text,
            "name": name,
            "position": position,
            "height": height,
            "weight": weight,
            "rating": rating,
            "high_school": high_school,
            "city": city,
            "state": state,
            "committed_to": committed_school,
            "url": abs_url
        }

    def _find_next_link(self, soup) -> Optional[str]:
        """
        Extremely robust pagination finder.
        Loops through ALL 'a' tags checking for keywords or specific classes.
        """
        # 1. Try standard CSS selectors for Next button
        selectors = [
            "a.pager-next",
            "li.next a",
            "a.next",
            ".rankings-page__pagination-link.next",
            "a.load-more",
            "a.load-more-btn"
        ]

        for selector in selectors:
            tag = soup.select_one(selector)
            if tag and tag.get("href"):
                # logger.info(f"Found next page via selector: {selector}") # Reduced noise
                return tag.get("href")

        # 2. Try searching by text content (case insensitive)
        # "Next", "Load More", ">"
        for a in soup.find_all("a", href=True):
            text = a.get_text(strip=True).lower()
            if text in ["next", "load more", "next »", ">", "next >"]:
                # logger.info(f"Found next page via text: '{text}'") # Reduced noise
                return a["href"]

        return None

    def run(self) -> List[Dict]:
        """Runs the scraper for the specific year and returns the data list."""
        current_url = self.base_url
        page_num = 1
        year_recruits = []

        logger.info(f"--- Start Year: {self.year} ---")

        while current_url:
            if MAX_PAGES and page_num > MAX_PAGES:
                logger.info("Max page limit reached.")
                break

            # Check for player count limit
            if len(year_recruits) >= MAX_PLAYERS_PER_YEAR:
                logger.info(f"Reached limit of {MAX_PLAYERS_PER_YEAR} players for {self.year}.")
                break

            # Reduced logging noise
            if page_num % 5 == 0:
                print(f"  {self.year}: Processing Page {page_num}...")

            html = self.fetch_page(current_url)
            if not html:
                logger.warning(f"Skipping page {page_num} due to fetch error.")
                break

            soup = BeautifulSoup(html, "html.parser")

            # --- Extract Items ---
            recruit_items = soup.select("li.rankings-page__list-item")
            if not recruit_items:
                recruit_items = soup.select(".rankings-page__list-item")

            if not recruit_items:
                logger.warning(f"No recruit items found on {current_url}.")
                break # Stop if empty page

            for item in recruit_items:
                try:
                    if "rankings-page__list-item--header" in item.get("class", []):
                        continue

                    # Stop midway if we hit the limit exactly
                    if len(year_recruits) >= MAX_PLAYERS_PER_YEAR:
                         break

                    data = self.parse_recruit_element(item)
                    if data.get("name"):
                        # Generate Unique ID
                        # Format: YYYY + 5 digit index (e.g. 201500001)
                        idx = len(year_recruits) + 1
                        player_id = int(f"{self.year}{idx:05d}")

                        # Reconstruct dict with player_id first
                        ordered_data = {"player_id": player_id}
                        ordered_data.update(data)

                        year_recruits.append(ordered_data)
                except Exception as e:
                    logger.error(f"Error parsing recruit: {e}")

            # --- Pagination Strategy ---
            next_href = self._find_next_link(soup)

            if next_href and len(year_recruits) < MAX_PLAYERS_PER_YEAR:
                current_url = urljoin(self.base_url, next_href)
                page_num += 1
            else:
                current_url = None

        logger.info(f"Year {self.year} stats: Found {len(year_recruits)} recruits across {page_num} pages.")
        return year_recruits

In [ ]:
# --- Main Execution ---

try:
    for year in range(START_YEAR, END_YEAR + 1):
        print(f"\n--- Starting Year: {year} ---")

        # Define output file for THIS SPECIFIC YEAR
        year_output_file = os.path.join(OUTPUT_FOLDER, f"recruits_{year}.csv")

        # Explicitly delete existing file to ensure a fresh start as requested
        if os.path.exists(year_output_file):
            try:
                os.remove(year_output_file)
                print(f"Deleted existing file: {year_output_file}")
            except OSError as e:
                print(f"Error deleting file {year_output_file}: {e}")

        # Construct URL for the specific year
        start_url = f"https://247sports.com/Season/{year}-Football/CompositeRecruitRankings/"

        scraper = RobustScraper(base_url=start_url, cache_dir=CACHE_DIR, year=year)
        year_data = scraper.run()

        # Save independent file for this year
        if year_data:
            df = pd.DataFrame(year_data)
            # Cleanup duplicates
            if "url" in df.columns:
                df.drop_duplicates(subset=["url"], keep="last", inplace=True)

            try:
                df.to_csv(year_output_file, index=False, encoding="utf-8-sig")
                logger.info(f"SAVED FILE: {year_output_file}")
                print(f"SAVED: {len(df)} records for {year} -> {year_output_file}")
            except Exception as e:
                logger.error(f"Failed to save file for {year}: {e}")
        else:
             logger.warning(f"No data found for year {year}")

        # Brief pause between years
        time.sleep(random.uniform(2, 5))

    print("\nAll years completed.")

except KeyboardInterrupt:
    logger.info("Scraper stopped by user.")
    print("Scraper stopped.")
except Exception as e:
    logger.critical(f"Fatal error in main loop: {e}", exc_info=True)

2026-01-20 19:10:06,570 - INFO - --- Start Year: 2015 ---



--- Starting Year: 2015 ---
  2015: Processing Page 5...
  2015: Processing Page 10...
  2015: Processing Page 15...
  2015: Processing Page 20...
  2015: Processing Page 25...
  2015: Processing Page 30...
  2015: Processing Page 35...
  2015: Processing Page 40...
  2015: Processing Page 45...
  2015: Processing Page 50...
  2015: Processing Page 55...
  2015: Processing Page 60...
  2015: Processing Page 65...
  2015: Processing Page 70...


2026-01-20 19:15:02,273 - WARNING - No recruit items found on https://247sports.com/season/2015-football/compositerecruitrankings/?ViewPath=~%2FViews%2FSkyNet%2FPlayerSportRanking%2F_SimpleSetForSeason.ascx&Page=73.
2026-01-20 19:15:02,275 - INFO - Year 2015 stats: Found 3566 recruits across 73 pages.
2026-01-20 19:15:02,316 - INFO - SAVED FILE: C:\Users\shave\OneDrive\Desktop\Football Recruit Data\recruits_2015.csv


SAVED: 3518 records for 2015 -> C:\Users\shave\OneDrive\Desktop\Football Recruit Data\recruits_2015.csv


2026-01-20 19:15:06,627 - INFO - --- Start Year: 2016 ---


...

2026-01-20 19:26:39,493 - WARNING - No recruit items found on https://247sports.com/season/2017-football/compositerecruitrankings/?ViewPath=~%2FViews%2FSkyNet%2FPlayerSportRanking%2F_SimpleSetForSeason.ascx&Page=87.
2026-01-20 19:26:39,494 - INFO - Year 2017 stats: Found 4271 recruits across 87 pages.
2026-01-20 19:26:39,563 - INFO - SAVED FILE: C:\Users\shave\OneDrive\Desktop\Football Recruit Data\recruits_2017.csv


SAVED: 4212 records for 2017 -> C:\Users\shave\OneDrive\Desktop\Football Recruit Data\recruits_2017.csv


2026-01-20 19:26:41,876 - INFO - --- Start Year: 2018 ---



--- Starting Year: 2018 ---
  2018: Processing Page 5...
  2018: Processing Page 10...
  2018: Processing Page 15...
  2018: Processing Page 20...
  2018: Processing Page 25...


Exception ignored from cffi callback <function buffer_callback at 0x000001DB847B76A0>:
Traceback (most recent call last):
  File "c:\Users\shave\AppData\Local\Programs\Python\Python311\Lib\site-packages\curl_cffi\curl.py", line 101, in buffer_callback
    @ffi.def_extern()
    
KeyboardInterrupt: 
2026-01-20 19:28:31,798 - ERROR - Request failed for https://247sports.com/season/2018-football/compositerecruitrankings/?ViewPath=~%2FViews%2FSkyNet%2FPlayerSportRanking%2F_SimpleSetForSeason.ascx&Page=26: Failed to perform, curl: (23) Failure writing output to destination, passed 13 returned 0. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
